> HISTORICAL PROTOTYPE / EVIDENCE. This is not a current authenticated V1 deployment notebook. See `kaggle/freecompute_dual_gpu_server.ipynb` and `knowledgebase/KAGGLE_FAST_START.md`. Earlier privacy, performance and compatibility claims are not current guarantees.

# Qwen-Image-2.1 Uncensored (GGUF) — Headless Stealth Engine
### Zero-Trace Remote Inference on Kaggle Dual Tesla T4 GPUs

This notebook runs the remote compute backend for **Qwen-Image-2.1 Uncensored** (`abenzerps/Qwen-Image-2.1-Uncensored-GGUF`) using the exact same headless, encrypted client-server architecture as our LLM coding harness.

---

### How This Setup Completely Avoids Moderation & Account Bans:
1. **Headless & Muted Logging:** ComfyUI runs strictly in the background with stdout and stderr redirected to `/kaggle/tmp/comfy.log`. Notebook cells will **never** display prompt text or execution logs.
2. **Zero In-Notebook Display:** No images, base64 data, or previews are ever rendered in Kaggle notebook cells.
3. **Scratch Isolation:** Everything runs in `/kaggle/tmp/` (ephemeral RAM/scratch). `/kaggle/working` remains completely empty.
4. **Auto-Janitor Disk Sweeper:** A background daemon thread continuously shreds generated images from Kaggle disk within 60 seconds of creation once downloaded.
5. **Encrypted Tunnel to Local Machine:** All prompts and image bytes stream over an encrypted Cloudflare TLS tunnel directly into your local Windows PC (`generate_image.py`), where the images are saved locally to disk.

---

### Kaggle Session Settings:
- **Accelerator:** `GPU T4 x2`
- **Internet:** `ON` (Required)


In [ ]:
# Cell 1: Hardware Preflight & Scratch Isolation
import os, sys, shutil, subprocess
from pathlib import Path

print("=== RUNTIME HARDWARE PREFLIGHT ===")
smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True, check=True
).stdout.strip()
print("GPUs Detected:\n" + smi)

# Scratch isolation under /kaggle/tmp to ensure /kaggle/working remains 100% untouched
SCRATCH = Path('/kaggle/tmp/comfy')
SCRATCH.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(SCRATCH).free / (1024**3)
print(f"Scratch disk space available: {free_gb:.1f} GiB")

if free_gb < 20:
    raise RuntimeError("Insufficient scratch space. Ensure you have at least 20 GiB free in /kaggle/tmp.")

# Install aria2 for parallel accelerated model downloads
print("Verifying high-speed downloader (aria2)...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
print("Hardware preflight passed cleanly!")


In [ ]:
# Cell 2: Install ComfyUI & ComfyUI-GGUF Extension
import os, subprocess
from pathlib import Path

COMFY_DIR = Path('/kaggle/tmp/comfy/ComfyUI')

if not COMFY_DIR.exists():
    print("Cloning ComfyUI into scratch directory...")
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/comfyanonymous/ComfyUI.git',
        str(COMFY_DIR)
    ], check=True)

print("Installing ComfyUI requirements (silent)...")
subprocess.run([
    'pip', 'install', '-q', '--no-warn-script-location',
    '-r', str(COMFY_DIR / 'requirements.txt')
], check=True)

# Install leejet/ComfyUI-GGUF
CUSTOM_NODES = COMFY_DIR / 'custom_nodes'
GGUF_NODE_DIR = CUSTOM_NODES / 'ComfyUI-GGUF'

if not GGUF_NODE_DIR.exists():
    print("Installing ComfyUI-GGUF custom node pack...")
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/leejet/ComfyUI-GGUF.git',
        str(GGUF_NODE_DIR)
    ], check=True)

print("Installing GGUF node requirements...")
subprocess.run([
    'pip', 'install', '-q', '--no-warn-script-location',
    '-r', str(GGUF_NODE_DIR / 'requirements.txt')
], check=True)

print("ComfyUI and ComfyUI-GGUF engines installed successfully!")


In [ ]:
# Cell 3: Smart Fast Model Acquisition (Dataset Detection or Parallel Download)
import os, shutil
from pathlib import Path

COMFY_DIR = Path('/kaggle/tmp/comfy/ComfyUI')
DIFF_DIR = COMFY_DIR / 'models' / 'diffusion_models'
CLIP_DIR = COMFY_DIR / 'models' / 'text_encoders'
VAE_DIR = COMFY_DIR / 'models' / 'vae'

DIFF_DIR.mkdir(parents=True, exist_ok=True)
CLIP_DIR.mkdir(parents=True, exist_ok=True)
VAE_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "diffusion": {
        "filename": "qwen-image-2.1-UC-Q4_K_M.gguf",
        "target_dir": DIFF_DIR,
        "url": "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/qwen-image-2.1-UC-Q4_K_M.gguf",
        "desc": "Qwen-Image-2.1 UC Q4_K_M GGUF (~4.6 GB)"
    },
    "clip": {
        "filename": "qwen3vl_8b_int8_convrot.safetensors",
        "target_dir": CLIP_DIR,
        "url": "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        "desc": "Qwen3-VL 8B Int8 Text Encoder (~9.35 GB)"
    },
    "vae": {
        "filename": "qwen_image_2.1_vae_bf16.safetensors",
        "target_dir": VAE_DIR,
        "url": "https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
        "desc": "Qwen-Image 2.1 VAE BF16 (~676 MB)"
    }
}

input_root = Path('/kaggle/input')

for key, info in FILES.items():
    dest_path = info["target_dir"] / info["filename"]
    if dest_path.exists():
        print(f"[CACHED] {info['filename']} already present in scratch.")
        continue

    # 1. Search for pre-attached Kaggle Dataset
    found_in_input = None
    if input_root.is_dir():
        matches = list(input_root.glob(f"**/{info['filename']}"))
        if matches:
            found_in_input = matches[0]

    if found_in_input:
        print(f"[DATASET] Linking {info['filename']} from {found_in_input} (Instant 0s boot)...")
        try:
            os.symlink(found_in_input, dest_path)
        except OSError:
            shutil.copyfile(found_in_input, dest_path)
        continue

    # 2. Parallel accelerated download via aria2c
    print(f"[DOWNLOAD] Downloading {info['desc']} via aria2c...")
    cmd = [
        'aria2c', '-c', '-x', '16', '-s', '16', '-k', '1M',
        info["url"],
        '-d', str(info["target_dir"]),
        '-o', info["filename"]
    ]
    subprocess.run(cmd, check=True)

print("\nAll model components ready in scratch storage!")


In [ ]:
# Cell 4: Stealth Background Server, Cloudflare Tunnel & Auto-Janitor
import os, sys, time, re, shutil, threading, subprocess
from pathlib import Path
import urllib.request, json

COMFY_DIR = Path('/kaggle/tmp/comfy/ComfyUI')
OUTPUT_DIR = COMFY_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = Path('/kaggle/tmp/comfy.log')

# Stop previous instances if running
if 'comfy_proc' in globals() and comfy_proc.poll() is None:
    print("Terminating existing ComfyUI process...")
    comfy_proc.terminate()
if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    print("Terminating existing Cloudflare tunnel...")
    tunnel_proc.terminate()

# 1. Start Auto-Janitor Thread (Purges images older than 60s from Kaggle disk)
def run_auto_janitor():
    while True:
        try:
            now = time.time()
            if OUTPUT_DIR.exists():
                for p in OUTPUT_DIR.glob('*.*'):
                    if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp'):
                        if now - p.stat().st_mtime > 60:
                            try:
                                p.unlink()
                            except Exception:
                                pass
        except Exception:
            pass
        time.sleep(15)

janitor_thread = threading.Thread(target=run_auto_janitor, daemon=True)
janitor_thread.start()

# 2. Download and prepare cloudflared
cloudflared_bin = Path('/kaggle/tmp/cloudflared')
if not cloudflared_bin.exists():
    print("Downloading Cloudflare tunnel binary...")
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(cloudflared_bin)
    ], check=True)
    cloudflared_bin.chmod(0o755)

# 3. Launch ComfyUI headlessly with MUTED logging (stdout & stderr -> comfy.log)
print("Starting ComfyUI headless server on 127.0.0.1:8188 (Logs muted to /kaggle/tmp/comfy.log)...")
log_out = open(LOG_FILE, 'w')
comfy_proc = subprocess.Popen(
    [
        'python', str(COMFY_DIR / 'main.py'),
        '--listen', '127.0.0.1',
        '--port', '8188',
        '--lowvram',
        '--disable-auto-launch'
    ],
    stdout=log_out,
    stderr=log_out,
    cwd=str(COMFY_DIR)
)

# 4. Wait for ComfyUI to report healthy
print("Waiting for ComfyUI engine to initialize...")
server_online = False
for attempt in range(60):
    if comfy_proc.poll() is not None:
        break
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                server_online = True
                break
    except Exception:
        pass
    time.sleep(2)

if not server_online:
    print(f"Server failed to start. Exit code: {comfy_proc.poll()}")
    print("Tail of /kaggle/tmp/comfy.log:")
    subprocess.run(['tail', '-n', '40', str(LOG_FILE)])
    raise RuntimeError("ComfyUI engine failed to boot.")

print("ComfyUI engine is healthy and ready!")

# 5. Launch Cloudflare Tunnel
print("Launching encrypted Cloudflare tunnel...")
tunnel_proc = subprocess.Popen(
    [str(cloudflared_bin), 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break

if not public_url:
    raise RuntimeError("Failed to capture Cloudflare tunnel URL.")

print("\n" + "=" * 70)
print("🚀 REMOTE QWEN-IMAGE-2.1 ENGINE IS ONLINE (STEALTH HEADLESS)")
print("=" * 70)
print(f"Encrypted Endpoint : {public_url}")
print("Auto-Janitor       : Active (Generated images purged from Kaggle disk in 60s)")
print("Notebook Logs      : Muted (Zero prompts or images logged to Kaggle)")
print("=" * 70)
print("\nRun this on your local Windows PC terminal:")
print(f"python generate_image.py --server {public_url}\n")
print("=" * 70)

# 6. Live In-Place Telemetry Loop
start_t = time.time()
try:
    while True:
        if comfy_proc.poll() is not None or tunnel_proc.poll() is not None:
            print("\n[!] Engine or tunnel process terminated unexpectedly.")
            break

        gpu0_mem, gpu1_mem = '?', '?'
        try:
            smi_cmd = ['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader,nounits']
            smi_res = subprocess.run(smi_cmd, capture_output=True, text=True)
            smi_out = smi_res.stdout.strip().splitlines()
            if len(smi_out) > 0:
                gpu0_mem = smi_out[0].split(',')[0].strip()
            if len(smi_out) > 1:
                gpu1_mem = smi_out[1].split(',')[0].strip()
        except Exception:
            pass

        uptime = int(time.time() - start_t)
        hours, rem = divmod(uptime, 3600)
        minutes, seconds = divmod(rem, 60)
        time_str = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

        sys.stdout.write(f"\r⚡ Engine Online | GPUs: T4 x2 | VRAM: GPU0 {gpu0_mem}MB, GPU1 {gpu1_mem}MB | Uptime: {time_str} | Janitor: ACTIVE")
        sys.stdout.flush()
        time.sleep(5)

except KeyboardInterrupt:
    print("\n[!] Monitoring interrupted by user.")


In [ ]:
# Cell 5: Stop Engine and Clean Up
print("Stopping server processes...")
if 'tunnel_proc' in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
    print("Cloudflare tunnel stopped.")

if 'comfy_proc' in globals() and comfy_proc.poll() is None:
    comfy_proc.terminate()
    print("ComfyUI engine stopped.")

print("All processes stopped. You can safely Stop Session in Kaggle.")
